In [531]:
import json
import shutil
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd

import altair as alt
import matplotlib.pyplot as plt
import plotly.express as px
import vl_convert
import kaleido

In [532]:
# Notebook is inside I2R/notebooks
PROJECT = Path("..").resolve()

DATA_RAW = PROJECT / "data" / "raw"
DATA_PROCESSED = PROJECT / "data" / "processed"
OUTPUTS = PROJECT / "outputs"

OUT_ROOT = OUTPUTS / "generated" / "pie"

CLEAR_OUTPUT = True

if CLEAR_OUTPUT and OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

LIBRARIES = ["altair", "matplotlib", "plotly"]
SUBDIRS = ["images", "tables", "meta"]

for sub in SUBDIRS:
    for lib in LIBRARIES:
        (OUT_ROOT / sub / lib).mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT)
print("Raw data path:", DATA_RAW)
print("Output path:", OUT_ROOT)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

print("Paths ready.")

Project root: C:\Users\Michelle\I2R
Raw data path: C:\Users\Michelle\I2R\data\raw
Output path: C:\Users\Michelle\I2R\outputs\generated\pie
Paths ready.


In [533]:
# Load csv train

DATA_TRAIN = PROJECT / "data" / "train"

csv_train = DATA_TRAIN / "Warehouse_and_Retail_Sales.csv"

df_train = pd.read_csv(csv_train)

df_train.head()

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
0,2020,1,REPUBLIC NATIONAL DISTRIBUTING CO,100009,BOOTLEG RED - 750ML,WINE,0.00,0.0,2.0
1,2020,1,PWSWN INC,100024,MOMENT DE PLAISIR - 750ML,WINE,0.00,1.0,4.0
2,2020,1,RELIABLE CHURCHILL LLLP,1001,S SMITH ORGANIC PEAR CIDER - 18.7OZ,BEER,0.00,0.0,1.0
3,2020,1,LANTERNA DISTRIBUTORS INC,100145,SCHLINK HAUS KABINETT - 750ML,WINE,0.00,0.0,1.0
4,2020,1,DIONYSOS IMPORTS INC,100293,SANTORINI GAVALA WHITE - 750ML,WINE,0.82,0.0,0.0


In [534]:
# date

# Create date column (first day of month)
df_train["date"] = pd.to_datetime(
    dict(year=df_train["YEAR"].astype("Int64"), month=df_train["MONTH"].astype("Int64"), day=1),
    errors="coerce"
)

# pie chart


In [535]:
# define chart id + meta helper

import json
from datetime import datetime
from uuid import uuid4

def new_chart_id(prefix="pie"):
    # short but unique
    return f"{prefix}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"

def save_metadata(meta: dict, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

In [536]:
# parameter statistics helper function
# TODO in Excel: Title color
# TODO in Excel: Title size
# TODO in Excel: Subtitle present?

PARAM_STATS = {
    "title_present": {
        True: 0.80,
        False: 0.20,
    },
    "title_color": {
        "black": 0.70,
        "darkblue": 0.15,
        "darkred": 0.10,
        "gray": 0.05,
    },
    "title_size": {
        "small": 0.20,
        "medium": 0.55,
        "large": 0.25,
    },
    "subtitle_present": {
        True: 0.25,
        False: 0.75,
    },
    "legend_present": {
        True: 0.55,
        False: 0.45,
    },
    "legend_title_present": {
        True: 0.50,
        False: 0.50,
    },
    "legend_title_color": {
        "black": 0.75,
        "gray": 0.15,
        "darkblue": 0.10,
    },
    "legend_outline": {
        True: 0.20,
        False: 0.80,
    },
    "legend_orient": {
        "right": 0.70,
        "bottom": 0.20,
        "left": 0.05,
        "top": 0.05,
    },
    "slice_text_present": {
        True: 0.65,
        False: 0.35,
    },
    "slice_text_content": {
        "label": 0.12,
        "percent": 0.18,
        "label+percent": 0.20,
        "value": 0.12,
        "value+percent": 0.14,
        "value+label": 0.12,
        "value+label+percent": 0.12,
    },
    "slice_text_position": {
        "inside": 0.45,
        "outside_with_leader_lines": 0.20,
        "outside_without_leader_lines": 0.15,
        "mixed": 0.10,
        "none": 0.10,
    },
    "slice_text_orientation": { #new
    "horizontal": 0.75,
    "radial": 0.25,
    },
    
    "stroke_width": {
        0: 0.30,
        1: 0.55,
        2: 0.15,
    },
    "explode": { #never
        False: 1.0,
        True: 0.0,
    },
    "palette_type": {
        "categorical": 0.55,
        "monochrome": 0.15,
        "grayscale": 0.10,
        "sequential": 0.20,
    },
    "sort_order": {
        "descending": 0.60,
        "ascending": 0.10,
        "unsorted": 0.30,
    },
    "percent_format": {
        "integer": 0.65,
        "one_decimal": 0.35,
    },
    "image_outline": { #outliens the piechart itself, not the full image
        False: 1.0,
        True: 0.0,
    },
    "slice_count_bucket": {
        "2": 0.08,
        "3-5": 0.40,
        "6-10": 0.32,
        "11-15": 0.15,
        "15+": 0.05,
    },
}



In [537]:
# weighted sampling helper function 

def weighted_choice(rng: np.random.Generator, options: dict):
    values = list(options.keys())
    probs = np.array(list(options.values()), dtype=float)
    probs = probs / probs.sum()
    return rng.choice(values, p=probs)

def sample_pie_style(rng: np.random.Generator, param_stats: dict):
    style = {
        "title_present": weighted_choice(rng, param_stats["title_present"]),
        "title_color": weighted_choice(rng, param_stats["title_color"]),
        "title_size": weighted_choice(rng, param_stats["title_size"]),
        "subtitle_present": weighted_choice(rng, param_stats["subtitle_present"]),
        "legend_present": weighted_choice(rng, param_stats["legend_present"]),
        "legend_title_present": weighted_choice(rng, param_stats["legend_title_present"]),
        "legend_title_color": weighted_choice(rng, param_stats["legend_title_color"]),
        "legend_outline": weighted_choice(rng, param_stats["legend_outline"]),
        "legend_orient": weighted_choice(rng, param_stats["legend_orient"]),
        "slice_text_present": weighted_choice(rng, param_stats["slice_text_present"]),
        "slice_text_content": weighted_choice(rng, param_stats["slice_text_content"]),
        "slice_text_position": weighted_choice(rng, param_stats["slice_text_position"]),
        "stroke_width": weighted_choice(rng, param_stats["stroke_width"]),
        "explode": weighted_choice(rng, param_stats["explode"]),
        "palette_type": weighted_choice(rng, param_stats["palette_type"]),
        "sort_order": weighted_choice(rng, param_stats["sort_order"]),
        "percent_format": weighted_choice(rng, param_stats["percent_format"]),
        "image_outline": weighted_choice(rng, param_stats["image_outline"]),
        "slice_count_bucket": weighted_choice(rng, param_stats["slice_count_bucket"]),
        "slice_text_orientation": weighted_choice(rng, param_stats["slice_text_orientation"]),
    }

    if not style["slice_text_present"]:
        style["slice_text_position"] = "none"
        style["slice_text_content"] = "none"

    if style["slice_text_position"] == "none":
        style["slice_text_content"] = "none"

    if not style["legend_present"]:
        style["legend_title_present"] = False
        style["legend_outline"] = False

    # Prevent visually bad combinations
    crowded_bucket = style["slice_count_bucket"] in ["11-15", "15+"]
    medium_bucket = style["slice_count_bucket"] in ["6-10"]

    if crowded_bucket:
        # inside text is too risky for many slices
        if style["slice_text_position"] == "inside":
            style["slice_text_position"] = "none"

        # very dense charts should avoid subtitles
        style["subtitle_present"] = False

        # legends at top/bottom are likely to collide
        if style["legend_orient"] in ["top", "bottom"]:
            style["legend_orient"] = "right"

    if medium_bucket:
        if style["slice_text_position"] == "inside" and style["slice_text_content"] in [
            "value+label",
            "value+label+percent",
            "label+percent",
        ]:
            style["slice_text_position"] = "outside_without_leader_lines"

    # Radial text only makes sense inside slices
    if style["slice_text_orientation"] == "radial" and style["slice_text_position"] != "inside":
        style["slice_text_orientation"] = "horizontal"

    if style["slice_count_bucket"] in ["6-10", "11-15", "15+"]:
        if style["slice_text_position"] == "inside":
            style["slice_text_position"] = "none"

     # Long text combos are risky for dense pies
    if style["slice_count_bucket"] in ["6-10", "11-15", "15+"]:
        if style["slice_text_content"] in [
            "value+label+percent",
            "value+label",
            "value+percent",
            "label+percent",
        ]:
            if style["slice_text_position"] == "inside":
                style["slice_text_position"] = "none"
                style["slice_text_content"] = "none"

    # Radial only really makes sense for inside labels
    if style["slice_text_orientation"] == "radial" and style["slice_text_position"] != "inside":
        style["slice_text_orientation"] = "horizontal"

    return style

In [538]:
def apply_sort_order(plot_df, value_col, style, rng=None):
    sort_order = style.get("sort_order", "descending")

    if sort_order == "descending":
        return plot_df.sort_values(value_col, ascending=False).reset_index(drop=True)

    elif sort_order == "ascending":
        return plot_df.sort_values(value_col, ascending=True).reset_index(drop=True)

    elif sort_order == "unsorted":
        if rng is None:
            return plot_df.sample(frac=1).reset_index(drop=True)
        return plot_df.sample(frac=1, random_state=int(rng.integers(0, 1_000_000))).reset_index(drop=True)

    return plot_df.reset_index(drop=True)


def get_percent_autopct(style):
    percent_format = style.get("percent_format", "one_decimal")

    if percent_format == "integer":
        return "%1.0f%%"
    elif percent_format == "one_decimal":
        return "%1.1f%%"
    else:
        return "%1.1f%%"


def get_altair_color_scale(style):
    palette_type = style.get("palette_type", "categorical")

    if palette_type == "categorical":
        return alt.Scale(scheme="category10")
    elif palette_type == "monochrome":
        return alt.Scale(range=["#cfe8ff", "#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#084594"])
    elif palette_type == "grayscale":
        return alt.Scale(range=["#f0f0f0", "#d9d9d9", "#bdbdbd", "#969696", "#636363", "#252525"])
    elif palette_type == "sequential":
        return alt.Scale(scheme="blues")
    else:
        return alt.Scale(scheme="category10")


def get_matplotlib_colors(n, style):
    palette_type = style.get("palette_type", "categorical")

    if palette_type == "categorical":
        cmap = plt.get_cmap("tab10")
        return [cmap(i % 10) for i in range(n)]
    elif palette_type == "monochrome":
        cmap = plt.get_cmap("Blues")
        return [cmap(0.25 + 0.65 * i / max(1, n - 1)) for i in range(n)]
    elif palette_type == "grayscale":
        cmap = plt.get_cmap("Greys")
        return [cmap(0.2 + 0.65 * i / max(1, n - 1)) for i in range(n)]
    elif palette_type == "sequential":
        cmap = plt.get_cmap("viridis")
        return [cmap(0.15 + 0.75 * i / max(1, n - 1)) for i in range(n)]
    else:
        cmap = plt.get_cmap("tab10")
        return [cmap(i % 10) for i in range(n)]


def get_plotly_colors(style):
    palette_type = style.get("palette_type", "categorical")

    if palette_type == "categorical":
        return px.colors.qualitative.Plotly
    elif palette_type == "monochrome":
        return ["#cfe8ff", "#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#084594"]
    elif palette_type == "grayscale":
        return ["#f0f0f0", "#d9d9d9", "#bdbdbd", "#969696", "#636363", "#252525"]
    elif palette_type == "sequential":
        return px.colors.sequential.Blues
    else:
        return px.colors.qualitative.Plotly


def get_outline_settings(style):
    outline_chart = bool(style.get("outline_chart", False))
    stroke_width = style.get("stroke_width", 1)

    if outline_chart:
        return {
            "edgecolor": "black",
            "linewidth": max(1.5, stroke_width + 1),
        }
    else:
        if stroke_width > 0:
            return {
                "edgecolor": "white",
                "linewidth": stroke_width,
            }
        else:
            return {
                "edgecolor": None,
                "linewidth": 0,
            }

In [539]:
# full-image border helper

def apply_image_outline_matplotlib(fig, ax, style):
    if not bool(style.get("image_outline", False)):
        return

    fig.patch.set_edgecolor("black")
    fig.patch.set_linewidth(2)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("black")
        spine.set_linewidth(1.5)

def apply_image_outline_plotly(fig, style):
    if not bool(style.get("image_outline", False)):
        return

    fig.update_layout(
        shapes=[
            dict(
                type="rect",
                xref="paper",
                yref="paper",
                x0=0,
                y0=0,
                x1=1,
                y1=1,
                line=dict(color="black", width=2),
                fillcolor="rgba(0,0,0,0)",
            )
        ]
    )

In [540]:
# helper functions for title/legend settings

def get_title_fontsize(style):
    size_map = {
        "small": 12,
        "medium": 16,
        "large": 20,
    }
    return size_map.get(style.get("title_size", "medium"), 16)


def get_altair_title_config(title, style):
    kwargs = {
        "text": title,
        "fontSize": get_title_fontsize(style),
        "color": style.get("title_color", "black"),
    }

    if style.get("subtitle_present", False):
        kwargs["subtitle"] = ["Generated chart"]

    return alt.TitleParams(**kwargs)


def get_plotly_title_font(style):
    return {
        "size": get_title_fontsize(style),
        "color": style.get("title_color", "black"),
    }


def build_slice_labels(plot_df, category_col, value_col, style):
    content = style.get("slice_text_content", "none")
    percent_format = style.get("percent_format", "one_decimal")

    total = plot_df[value_col].sum()
    pct_values = 100 * plot_df[value_col] / total

    if percent_format == "integer":
        pct_str = pct_values.round(0).astype(int).astype(str) + "%"
    else:
        pct_str = pct_values.round(1).astype(str) + "%"

    value_str = plot_df[value_col].round(1).astype(str)
    label_str = plot_df[category_col].astype(str)

    if content == "label":
        return label_str
    elif content == "percent":
        return pct_str
    elif content == "label+percent":
        return label_str + " | " + pct_str
    elif content == "value":
        return value_str
    elif content == "value+percent":
        return value_str + " | " + pct_str
    elif content == "value+label":
        return value_str + " | " + label_str
    elif content == "value+label+percent":
        return value_str + " | " + label_str + " | " + pct_str
    else:
        return pd.Series([""] * len(plot_df), index=plot_df.index)

In [541]:
# slice-count helper

def sample_slice_count_from_bucket(rng, bucket, available_count):
    if available_count < 2:
        return None

    if bucket == "2":
        low, high = 2, 2
    elif bucket == "3-5":
        low, high = 3, 5
    elif bucket == "6-10":
        low, high = 6, 10
    elif bucket == "11-15":
        low, high = 11, 15
    elif bucket == "15+":
        low, high = 16, min(25, available_count)
    else:
        low, high = 3, 8

    low = min(low, available_count)
    high = min(high, available_count)

    if low > high:
        return None

    return int(rng.integers(low, high + 1))

In [542]:
import numpy as np

def compute_altair_text_geometry(plot_df, value_col, pos, outer_radius=160, cx=200, cy=200):
    df = plot_df.copy().reset_index(drop=True)

    total = df[value_col].sum()
    frac = df[value_col] / total

    df["_start_angle"] = frac.cumsum().shift(fill_value=0) * 2 * np.pi
    df["_end_angle"] = frac.cumsum() * 2 * np.pi
    df["_mid_angle"] = (df["_start_angle"] + df["_end_angle"]) / 2

    # Altair pies start at top
    theta = df["_mid_angle"] - np.pi / 2

    if pos == "inside":
        r = outer_radius * 0.58
    elif pos == "outside_without_leader_lines":
        r = outer_radius * 1.18
    else:
        r = outer_radius * 0.58

    df["_label_x"] = cx + r * np.cos(theta)
    df["_label_y"] = cy + r * np.sin(theta)
    df["_cos"] = np.cos(theta)

    return df

In [543]:
def compute_altair_label_positions(plot_df, value_col, radius, cx=200, cy=200):
    df = plot_df.copy()

    total = df[value_col].sum()
    frac = df[value_col] / total

    # cumulative angles
    df["_start_angle"] = frac.cumsum().shift(fill_value=0) * 2 * np.pi
    df["_end_angle"] = frac.cumsum() * 2 * np.pi
    df["_mid_angle"] = (df["_start_angle"] + df["_end_angle"]) / 2

    # Vega-Lite pies start at 12 o'clock, so rotate by -pi/2
    theta = df["_mid_angle"] - np.pi / 2

    df["_label_x"] = cx + radius * np.cos(theta)
    df["_label_y"] = cy + radius * np.sin(theta)

    return df

In [544]:
def add_outside_labels_with_leader_lines(ax, wedges, labels, radius=1.0, text_radius=1.25):
    for wedge, label in zip(wedges, labels):
        angle = 0.5 * (wedge.theta1 + wedge.theta2)
        angle_rad = np.deg2rad(angle)

        x = np.cos(angle_rad) * radius
        y = np.sin(angle_rad) * radius

        tx = np.cos(angle_rad) * text_radius
        ty = np.sin(angle_rad) * text_radius

        ha = "left" if tx >= 0 else "right"

        ax.annotate(
            label,
            xy=(x, y),
            xytext=(tx, ty),
            ha=ha,
            va="center",
            arrowprops=dict(arrowstyle="-", lw=1, color="gray"),
            fontsize=9,
        )

In [545]:
# # ORIGINAL

# # pie data samples 

# #choose random month with pos total sales
# # group by item type (or later supplier)
# #keep top-k + other
# # reject degenerate cases


# def sample_pie_data(
#     df: pd.DataFrame,
#     rng: np.random.Generator,
#     date_col="date",
#     category_col="ITEM TYPE",
#     value_col="RETAIL SALES",
#     k_min=3,
#     k_max=8,
#     min_total=1.0,
# ):
#     # 1) choose a valid month (positive totals)
#     monthly_totals = df.groupby(date_col)[value_col].sum()
#     valid_months = monthly_totals[monthly_totals > min_total].index.to_numpy()
#     if len(valid_months) == 0:
#         raise ValueError("No valid months with positive totals found.")

#     month = pd.to_datetime(rng.choice(valid_months))

#     # 2) aggregate within month by category
#     s = (
#         df.loc[df[date_col] == month]
#           .groupby(category_col)[value_col]
#           .sum()
#           .sort_values(ascending=False)
#     )

#     # remove zeros / negatives (pie needs non-negative)
#     s = s[s > 0]

#     # Need enough categories
#     if len(s) < k_min:
#         return None  # signal "resample"

#     # 3) choose top_k
#     top_k = int(rng.integers(k_min, min(k_max, len(s)) + 1))
#     top = s.head(top_k).copy()
#     other_sum = s.iloc[top_k:].sum()

#     if other_sum > 0:
#         top.loc["Other"] = other_sum

#     # final sanity
#     total = float(top.sum())
#     if total <= min_total or len(top) < k_min:
#         return None

#     # return a plotting table + context
#     plot_df = top.reset_index()
#     plot_df.columns = [category_col, value_col]

#     context = {
#         "month": month,
#         "top_k": top_k,
#         "total": total,
#         "n_categories_raw": int(len(s)),
#     }
#     return plot_df, context

In [546]:
# DATA SAMPLER

def sample_pie_data(
    df: pd.DataFrame,
    rng: np.random.Generator,
    style: dict,
    date_col: str = "date",
    min_total: float = 1.0,
):
    value_col = rng.choice(
        ["RETAIL SALES", "WAREHOUSE SALES", "RETAIL TRANSFERS"],
        p=[0.5, 0.3, 0.2]
    )

    mode = rng.choice(
        ["month_itemtype", "month_supplier", "supplier_itemtype", "itemtype_supplier"],
        p=[0.30, 0.25, 0.20, 0.25]
    )

    window_months = int(rng.choice([1, 3, 12], p=[0.55, 0.30, 0.15]))

    monthly_totals = df.groupby(date_col)[value_col].sum()
    valid_months = monthly_totals[monthly_totals > min_total].index.to_numpy()

    if len(valid_months) == 0:
        raise ValueError("No valid months with positive totals found.")

    start = pd.to_datetime(rng.choice(valid_months))
    end = start + pd.DateOffset(months=window_months)

    dfw = df[(df[date_col] >= start) & (df[date_col] < end)].copy()

    context = {
        "mode": mode,
        "value_col": value_col,
        "window_months": window_months,
        "start": start,
        "end": end,
    }

    if mode == "month_itemtype":
        category_col = "ITEM TYPE"
        s = dfw.groupby(category_col)[value_col].sum().sort_values(ascending=False)

    elif mode == "month_supplier":
        category_col = "SUPPLIER"
        s = dfw.groupby(category_col)[value_col].sum().sort_values(ascending=False)

    elif mode == "supplier_itemtype":
        category_col = "ITEM TYPE"
        supplier = rng.choice(df["SUPPLIER"].dropna().unique())
        dfw = dfw[dfw["SUPPLIER"] == supplier]
        s = dfw.groupby(category_col)[value_col].sum().sort_values(ascending=False)
        context["filter_supplier"] = supplier

    else:
        category_col = "SUPPLIER"
        item_type = rng.choice(df["ITEM TYPE"].dropna().unique())
        dfw = dfw[dfw["ITEM TYPE"] == item_type]
        s = dfw.groupby(category_col)[value_col].sum().sort_values(ascending=False)
        context["filter_item_type"] = item_type

    s = s[s > 0]

    if len(s) < 2:
        return None

    desired_k = sample_slice_count_from_bucket(
        rng,
        style.get("slice_count_bucket", "3-5"),
        available_count=len(s)
    )

    if desired_k is None:
        return None

    top = s.head(desired_k).copy()

    other_sum = s.iloc[desired_k:].sum()
    if other_sum > 0:
        top.loc["Other"] = other_sum

    total = float(top.sum())
    if total <= min_total or len(top) < 2:
        return None

    shares = (top / total).values
    if shares.max() > 0.90:
        return None

    plot_df = top.reset_index()
    plot_df.columns = [category_col, value_col]

    context.update({
        "category_col": category_col,
        "n_slices": len(plot_df),
        "requested_slice_count_bucket": style.get("slice_count_bucket"),
        "total": total,
        "has_other_category": bool(other_sum > 0),
    })

    return plot_df, context

In [547]:
# title helper
# !!!!!!!!!!!!!!!!!!Maybe include LLM !!!!!!!!!!!!!!!!

def make_title(context: dict):
    value_col = context["value_col"]
    category_col = context["category_col"]
    start_str = pd.to_datetime(context["start"]).strftime("%Y-%m")
    return f"{value_col} share by {category_col} ({start_str})"

In [548]:
# altair specific label sanitiser

def sanitize_altair_text_layout(plot_df, style):
    style = dict(style)
    n = len(plot_df)

    pos = style.get("slice_text_position", "none")
    content = style.get("slice_text_content", "none")
    orientation = style.get("slice_text_orientation", "horizontal")

    # No labels requested
    if pos == "none" or content == "none":
        style["slice_text_position"] = "none"
        style["slice_text_content"] = "none"
        style["slice_text_orientation"] = "horizontal"
        return style

    # Altair cannot reliably do leader lines here
    if pos == "outside_with_leader_lines":
        style["slice_text_position"] = "outside_without_leader_lines"

    # Mixed is too unstable in this setup
    if pos == "mixed":
        if n <= 5:
            style["slice_text_position"] = "inside"
        else:
            style["slice_text_position"] = "none"
            style["slice_text_content"] = "none"

    # Dense pies: suppress labels entirely
    if n >= 8:
        style["slice_text_position"] = "none"
        style["slice_text_content"] = "none"
        style["slice_text_orientation"] = "horizontal"
        return style

    # Medium pies: only short inside labels
    if n >= 6:
        if pos == "inside" and content in [
            "value+label+percent",
            "value+label",
            "value+percent",
            "label+percent",
        ]:
            style["slice_text_position"] = "none"
            style["slice_text_content"] = "none"

    # Radial text only for inside labels and small pies
    if orientation == "radial":
        if style["slice_text_position"] != "inside" or n > 5:
            style["slice_text_orientation"] = "horizontal"

    return style


In [549]:
import numpy as np
import pandas as pd
import altair as alt

def compute_altair_leaderline_geometry(
    plot_df,
    value_col,
    outer_radius=160,
    cx=200,
    cy=200,
    radial_line_len=18,
    horizontal_line_len=22,
    label_pad=4,
):
    df = plot_df.copy().reset_index(drop=True)

    total = df[value_col].sum()
    frac = df[value_col] / total

    df["_start_angle"] = frac.cumsum().shift(fill_value=0) * 2 * np.pi
    df["_end_angle"] = frac.cumsum() * 2 * np.pi
    df["_mid_angle"] = (df["_start_angle"] + df["_end_angle"]) / 2

    theta = df["_mid_angle"] - np.pi / 2
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    df["_x0"] = cx + outer_radius * cos_t
    df["_y0"] = cy + outer_radius * sin_t

    df["_x1"] = cx + (outer_radius + radial_line_len) * cos_t
    df["_y1"] = cy + (outer_radius + radial_line_len) * sin_t

    right_side = cos_t >= 0
    df["_side"] = np.where(right_side, "right", "left")

    df["_x2"] = np.where(right_side, df["_x1"] + horizontal_line_len, df["_x1"] - horizontal_line_len)
    df["_y2"] = df["_y1"]

    df["_label_x"] = np.where(right_side, df["_x2"] + label_pad, df["_x2"] - label_pad)
    df["_label_y"] = df["_y2"]

    df["_angle"] = np.degrees(df["_mid_angle"]) - 90

    return df


def build_altair_pie_labels_with_leader_lines(
    label_df,
    label_col="custom_label",
    chart_width=400,
    chart_height=400,
    font_size=10,
    text_color="black",
    line_color="black",
):
    radial_lines = (
        alt.Chart(label_df)
        .mark_rule(color=line_color)
        .encode(
            x=alt.X("_x0:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
            y=alt.Y("_y0:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
            x2=alt.X2("_x1:Q"),
            y2=alt.Y2("_y1:Q"),
        )
    )

    horizontal_lines = (
        alt.Chart(label_df)
        .mark_rule(color=line_color)
        .encode(
            x=alt.X("_x1:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
            y=alt.Y("_y1:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
            x2=alt.X2("_x2:Q"),
            y2=alt.Y2("_y2:Q"),
        )
    )

    text_right = (
        alt.Chart(label_df)
        .transform_filter(alt.datum._side == "right")
        .mark_text(
            align="left",
            baseline="middle",
            size=font_size,
            color=text_color,
        )
        .encode(
            x=alt.X("_label_x:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
            y=alt.Y("_label_y:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
            text=alt.Text(f"{label_col}:N"),
        )
    )

    text_left = (
        alt.Chart(label_df)
        .transform_filter(alt.datum._side == "left")
        .mark_text(
            align="right",
            baseline="middle",
            size=font_size,
            color=text_color,
        )
        .encode(
            x=alt.X("_label_x:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
            y=alt.Y("_label_y:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
            text=alt.Text(f"{label_col}:N"),
        )
    )

    return radial_lines + horizontal_lines + text_right + text_left

In [550]:
import numpy as np

def get_matplotlib_text_rotation(mid_angle_deg, orientation):
    """
    Allowed orientations:
      - horizontal: always 0 degrees
      - radial: along the radius, kept upright
    """
    if orientation == "horizontal":
        return 0

    if orientation == "radial":
        rot = mid_angle_deg

        # keep text upright on the left half
        if 90 < rot < 270:
            rot += 180

        # optional normalization
        while rot > 180:
            rot -= 360
        while rot <= -180:
            rot += 360

        return rot

    return 0


def add_outside_labels_with_leader_lines(
    ax,
    wedges,
    labels,
    orientation="horizontal", #if you want outside labels with leader lines also to support radial, change this to orientation=orientation
    radius=1.0,
    text_radius=1.28,
    fontsize=9,
):
    for wedge, label in zip(wedges, labels):
        angle = 0.5 * (wedge.theta1 + wedge.theta2)
        angle_rad = np.deg2rad(angle)

        x = np.cos(angle_rad) * radius
        y = np.sin(angle_rad) * radius

        tx = np.cos(angle_rad) * text_radius
        ty = np.sin(angle_rad) * text_radius

        ha = "left" if tx >= 0 else "right"

        # outside labels should stay horizontal unless you explicitly want radial outside too
        rotation = 0 if orientation == "horizontal" else get_matplotlib_text_rotation(angle, "radial")

        ax.annotate(
            label,
            xy=(x, y),
            xytext=(tx, ty),
            ha=ha,
            va="center",
            rotation=rotation,
            rotation_mode="anchor",
            fontsize=fontsize,
            arrowprops=dict(arrowstyle="-", lw=0.8, color="gray"),
        )

In [551]:
# # ORIGINAL ALTAIR GENERATOR
# # generate altail charts and export to svg

# # plot function

# def make_pie_chart_altair(plot_df, category_col, value_col, title=None):
#     chart = (
#         alt.Chart(plot_df, title=title)
#         .mark_arc()
#         .encode(
#             theta=alt.Theta(field=value_col, type="quantitative"),
#             color=alt.Color(field=category_col, type="nominal"),
#             tooltip=[category_col, value_col]
#         )
#     )
#     return chart

In [552]:
def render_pie_altair(plot_df, category_col, value_col, title, style, rng=None):
    plot_df = apply_sort_order(plot_df, value_col, style, rng=rng).copy()
    style = sanitize_altair_text_layout(plot_df, style)

    total = plot_df[value_col].sum()
    plot_df["percentage"] = 100 * plot_df[value_col] / total
    plot_df["custom_label"] = build_slice_labels(plot_df, category_col, value_col, style)

    color_scale = get_altair_color_scale(style)

    legend = None
    if style["legend_present"]:
        legend = alt.Legend(
            orient=style["legend_orient"],
            title=category_col if style["legend_title_present"] else alt.Undefined,
            labelColor="black",
            titleColor=style["legend_title_color"],
        )

    if style["title_present"]:
        title_params = get_altair_title_config(title, style)
        base = alt.Chart(plot_df, title=title_params)
    else:
        base = alt.Chart(plot_df)

    arc = base.mark_arc(
        outerRadius=160,
        innerRadius=0,
        stroke="white" if style["stroke_width"] > 0 else None,
        strokeWidth=style["stroke_width"],
    ).encode(
        theta=alt.Theta(field=value_col, type="quantitative"),
        color=alt.Color(
            field=category_col,
            type="nominal",
            scale=color_scale,
            legend=legend,
        ),
        tooltip=[category_col, value_col, "percentage"],
    ).properties(
        width=400,
        height=400
    )

    pos = style.get("slice_text_position", "none")
    content = style.get("slice_text_content", "none")
    orientation = style.get("slice_text_orientation", "horizontal")

    if pos == "none" or content == "none":
        return arc.configure_view(stroke=None)

    # optional: downgrade unsupported combo
    if pos == "mixed":
        pos = "inside"

    # --- case 1: outside labels with leader lines ---
    if pos == "outside_with_leader_lines":
        label_df = compute_altair_leaderline_geometry(
            plot_df,
            value_col=value_col,
            outer_radius=160,
            cx=200,
            cy=200,
            radial_line_len=18,
            horizontal_line_len=22,
            label_pad=4,
        )

        labels_and_lines = build_altair_pie_labels_with_leader_lines(
            label_df,
            label_col="custom_label",
            chart_width=400,
            chart_height=400,
            font_size=10,
            text_color="black",
            line_color="black",
        )

        return (arc + labels_and_lines).configure_view(stroke=None)

    # --- case 2: inside / outside_without_leader_lines ---
    label_df = compute_altair_text_geometry(
        plot_df,
        value_col=value_col,
        pos=pos,
        outer_radius=160,
        cx=200,
        cy=200,
    )

    base_text = alt.Chart(label_df).encode(
        x=alt.X("_label_x:Q", scale=alt.Scale(domain=[0, 400]), axis=None),
        y=alt.Y("_label_y:Q", scale=alt.Scale(domain=[400, 0]), axis=None),
        text=alt.Text("custom_label:N"),
    )

    if orientation == "radial" and pos == "inside":
        base_text = base_text.encode(
            angle=alt.Angle("_angle:Q")
        )

    if pos == "inside":
        text = base_text.mark_text(
            size=10,
            color="black",
            baseline="middle",
            align="center",
        )
    else:
        text_right = base_text.transform_filter(
            alt.datum._cos >= 0
        ).mark_text(
            size=10,
            color="black",
            baseline="middle",
            align="left",
        )

        text_left = base_text.transform_filter(
            alt.datum._cos < 0
        ).mark_text(
            size=10,
            color="black",
            baseline="middle",
            align="right",
        )

        text = text_right + text_left

    return (arc + text).configure_view(stroke=None)

In [553]:
def sanitize_matplotlib_text_layout(plot_df, style):
    style = dict(style)
    n = len(plot_df)
    content = style.get("slice_text_content", "none")
    pos = style.get("slice_text_position", "none")

    # Very crowded pies: no direct text
    if n >= 12:
        style["slice_text_position"] = "none"
        style["slice_text_content"] = "none"
        return style

    # Crowded pies: keep only short outside labels or none
    if n >= 9:
        if pos == "inside":
            style["slice_text_position"] = "none"
            style["slice_text_content"] = "none"
        elif content in ["value+label+percent", "value+label", "label+percent", "value+percent"]:
            style["slice_text_content"] = "label"

    # Medium pies: avoid verbose inside labels
    if n >= 6:
        if pos == "inside" and content in ["value+label+percent", "value+label", "label+percent"]:
            style["slice_text_content"] = "percent"

    return style

In [554]:

def add_outside_labels_with_leader_lines(ax, wedges, labels, radius=1.0, text_radius=1.28, fontsize=9):
    for wedge, label in zip(wedges, labels):
        angle = 0.5 * (wedge.theta1 + wedge.theta2)
        angle_rad = np.deg2rad(angle)

        x = np.cos(angle_rad) * radius
        y = np.sin(angle_rad) * radius

        tx = np.cos(angle_rad) * text_radius
        ty = np.sin(angle_rad) * text_radius

        ha = "left" if tx >= 0 else "right"

        ax.annotate(
            label,
            xy=(x, y),
            xytext=(tx, ty),
            ha=ha,
            va="center",
            fontsize=fontsize,
            arrowprops=dict(arrowstyle="-", lw=0.8, color="gray"),
        )

In [555]:


def render_pie_matplotlib(plot_df, category_col, value_col, title, style, rng=None):
    plot_df = apply_sort_order(plot_df, value_col, style, rng=rng)
    style = sanitize_matplotlib_text_layout(plot_df, style)

    colors = get_matplotlib_colors(len(plot_df), style)
    custom_labels = build_slice_labels(plot_df, category_col, value_col, style)

    # hard-disable exploded slices
    style["explode"] = False
    explode = [0.0] * len(plot_df)

    wedgeprops = None
    if style["stroke_width"] > 0:
        wedgeprops = {
            "edgecolor": "white",
            "linewidth": style["stroke_width"],
        }

    pos = style.get("slice_text_position", "none")
    content = style.get("slice_text_content", "none")
    orientation = style.get("slice_text_orientation", "horizontal")
    n_slices = len(plot_df)

    if n_slices <= 5:
        figsize = (8, 6)
    elif n_slices <= 8:
        figsize = (10, 7)
    else:
        figsize = (12, 8)

    fig, ax = plt.subplots(figsize=figsize, facecolor="white")

    labels = None
    autopct = None
    labeldistance = 1.08
    pctdistance = 0.62
    radius = 1.0

    if n_slices >= 9:
        radius = 0.88
    elif n_slices >= 6:
        radius = 0.93

    use_manual_leaders = False

    if pos == "none" or content == "none":
        labels = None
        autopct = None

    elif pos == "inside":
        labels = None
        autopct = lambda pct: ""

    elif pos == "outside_with_leader_lines":
        labels = None
        autopct = None
        use_manual_leaders = True

    elif pos == "outside_without_leader_lines":
        labels = custom_labels
        autopct = None
        labeldistance = 1.08 if n_slices <= 6 else 1.14

    elif pos == "mixed":
        if n_slices <= 6:
            labels = plot_df[category_col]
            autopct = get_percent_autopct(style)
            labeldistance = 1.10
            pctdistance = 0.60
        else:
            labels = None
            autopct = None

    if autopct is None:
        wedges, texts = ax.pie(
            plot_df[value_col],
            labels=labels,
            explode=explode,
            colors=colors,
            wedgeprops=wedgeprops,
            startangle=90,
            labeldistance=labeldistance,
            pctdistance=pctdistance,
            radius=radius,
        )
        autotexts = []
    else:
        wedges, texts, autotexts = ax.pie(
            plot_df[value_col],
            labels=labels,
            autopct=autopct,
            explode=explode,
            colors=colors,
            wedgeprops=wedgeprops,
            startangle=90,
            labeldistance=labeldistance,
            pctdistance=pctdistance,
            radius=radius,
        )

    if pos == "inside" and content != "none":
        for i, t in enumerate(autotexts):
            t.set_text(custom_labels.iloc[i])

    # strictly enforce only horizontal or radial
    if pos == "inside":
        for wedge, t in zip(wedges, autotexts):
            angle = 0.5 * (wedge.theta1 + wedge.theta2)
            rotation = get_matplotlib_text_rotation(angle, orientation)

            t.set_rotation(rotation)
            t.set_rotation_mode("anchor")
            t.set_ha("center")
            t.set_va("center")
    else:
        for t in texts:
            t.set_rotation(0)
            t.set_rotation_mode("anchor")
        for t in autotexts:
            t.set_rotation(0)
            t.set_rotation_mode("anchor")

    if use_manual_leaders:
        add_outside_labels_with_leader_lines(
            ax,
            wedges,
            custom_labels,
            #orientation="horizontal",  # safest for outside labels
            radius=radius,
            text_radius=1.22 if n_slices <= 6 else 1.32,
            fontsize=9 if n_slices <= 6 else 8,
        )

    for t in texts:
        t.set_fontsize(10 if n_slices <= 6 else 8)
    for t in autotexts:
        t.set_fontsize(10 if n_slices <= 6 else 8)

    title_y = 0.965
    subtitle_y = 0.925

    if style["title_present"]:
        fig.suptitle(
            title,
            color=style["title_color"],
            fontsize=get_title_fontsize(style),
            y=title_y,
        )

    if style["subtitle_present"]:
        fig.text(
            0.5,
            subtitle_y,
            "Generated chart",
            ha="center",
            va="center",
            fontsize=11,
            color="gray", # change
        )

    legend = None
    if style["legend_present"]:
        legend_title = category_col if style["legend_title_present"] else None

        if style["legend_orient"] == "right":
            legend = ax.legend(
                wedges, plot_df[category_col],
                title=legend_title,
                loc="center left",
                bbox_to_anchor=(1.02, 0.5),
                frameon=style["legend_outline"],
            )
        elif style["legend_orient"] == "left":
            legend = ax.legend(
                wedges, plot_df[category_col],
                title=legend_title,
                loc="center right",
                bbox_to_anchor=(-0.08, 0.5),
                frameon=style["legend_outline"],
            )
        elif style["legend_orient"] == "top":
            legend = ax.legend(
                wedges, plot_df[category_col],
                title=legend_title,
                loc="lower center",
                bbox_to_anchor=(0.5, 1.02),
                ncol=2,
                frameon=style["legend_outline"],
            )
        elif style["legend_orient"] == "bottom":
            legend = ax.legend(
                wedges, plot_df[category_col],
                title=legend_title,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.08),
                ncol=2,
                frameon=style["legend_outline"],
            )

        if legend is not None and legend.get_title() is not None:
            legend.get_title().set_color(style["legend_title_color"])

    if style["legend_present"] and style["legend_orient"] == "right":
        plt.subplots_adjust(left=0.08, right=0.70, top=0.80, bottom=0.10)
    elif style["legend_present"] and style["legend_orient"] == "left":
        plt.subplots_adjust(left=0.30, right=0.95, top=0.80, bottom=0.10)
    elif style["legend_present"] and style["legend_orient"] == "top":
        plt.subplots_adjust(left=0.08, right=0.95, top=0.72, bottom=0.10)
    elif style["legend_present"] and style["legend_orient"] == "bottom":
        plt.subplots_adjust(left=0.08, right=0.95, top=0.80, bottom=0.24)
    else:
        plt.subplots_adjust(left=0.08, right=0.95, top=0.80, bottom=0.10)

    return fig

In [556]:
# Plotly renderer
def render_pie_plotly(plot_df, category_col, value_col, title, style, rng=None):
    plot_df = apply_sort_order(plot_df, value_col, style, rng=rng)

    color_sequence = get_plotly_colors(style)

    fig_kwargs = {
        "data_frame": plot_df,
        "names": category_col,
        "values": value_col,
        "color_discrete_sequence": color_sequence,
    }

    if style["title_present"]:
        fig_kwargs["title"] = title

    fig = px.pie(**fig_kwargs)

    fig.update_traces(
        marker=dict(
            line=dict(
                color="white" if style["stroke_width"] > 0 else None,
                width=style["stroke_width"],
            )
        )
    )

    pos = style.get("slice_text_position", "none")
    content = style.get("slice_text_content", "none")

    if pos == "none" or content == "none":
        fig.update_traces(textinfo="none")

    elif content == "label":
        fig.update_traces(textinfo="label")
    elif content == "percent":
        fig.update_traces(textinfo="percent")
    elif content == "label+percent":
        fig.update_traces(textinfo="label+percent")
    else:
        custom_text = build_slice_labels(plot_df, category_col, value_col, style)
        fig.update_traces(textinfo="text", text=custom_text)

    if pos == "inside":
        fig.update_traces(textposition="inside")
    elif pos in ["outside_with_leader_lines", "outside_without_leader_lines", "mixed"]:
        fig.update_traces(textposition="outside")

    legend_title_text = category_col if style["legend_title_present"] else ""

    fig.update_layout(
        showlegend=bool(style["legend_present"]),
        title_font=get_plotly_title_font(style),
        legend=dict(
            orientation="v" if style["legend_orient"] in ["left", "right"] else "h",
            title=dict(
                text=legend_title_text,
                font=dict(color=style["legend_title_color"])
            ),
            bordercolor="black" if style["legend_outline"] else "rgba(0,0,0,0)",
            borderwidth=1 if style["legend_outline"] else 0,
        ),
    )

    if style["subtitle_present"]:
        fig.add_annotation(
            text="Generated chart",
            x=0.5,
            y=1.02,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=12, color="gray")
        )

    apply_image_outline_plotly(fig, style)

    return fig

In [557]:
# ORIGINAL SAVE 
# # save svg

# def save_altair_svg(chart, svg_path):
#     svg_path = Path(svg_path)
#     svg_path.parent.mkdir(parents=True, exist_ok=True)
#     chart.save(str(svg_path), format="svg")

In [558]:
# # test for plotly
# WORKS NOW 

# import plotly.express as px
# from pathlib import Path

# test_df = px.data.tips().groupby("day", as_index=False)["total_bill"].sum()

# fig = px.pie(test_df, names="day", values="total_bill", title="Test Pie")

# test_path = Path("plotly_test.png")
# fig.write_image(str(test_path), format="png", scale=2)

# print("Saved to:", test_path.resolve())

In [559]:
# New save
def save_altair_svg(chart, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    chart.save(str(out_path), format="svg")


# def save_plotly_html(fig, out_path: Path):
#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     fig.write_html(str(out_path))

def save_plotly_png(fig, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(out_path), format="png", scale=2)

def save_matplotlib_png(fig, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

In [560]:
# # OLD UNIFIED GENERATOR
# def generate_pie(
#     df: pd.DataFrame,
#     out_root: Path,
#     dataset_source: str,
#     rng_seed: int = 42,
#     date_col="date",
#     category_col="ITEM TYPE",
#     value_col="RETAIL SALES",
#     max_tries: int = 50,
# ):
#     rng = np.random.default_rng(rng_seed)
#     chart_id = new_chart_id("pie")

#     for attempt in range(1, max_tries + 1):
#         sampled = sample_pie_data(
#             df=df,
#             rng=rng,
#             date_col=date_col,
#             category_col=category_col,
#             value_col=value_col,
#         )
#         if sampled is None:
#             continue

#         plot_df, context = sampled

#         # --- Save the table used for plotting
#         table_path = out_root / "tables" / f"{chart_id}.csv"
#         plot_df.to_csv(table_path, index=False)

#         # --- Build & save chart
#         month_str = pd.to_datetime(context["month"]).strftime("%Y-%m")
#         title = f"{value_col} share by {category_col} ({month_str})"
#         chart = make_pie_chart_altair(plot_df, category_col, value_col, title=title)

#         svg_path = out_root / "images" / f"{chart_id}.svg"
#         save_altair_svg(chart, svg_path)

#         # --- Metadata
#         meta = {
#             "chart_id": chart_id,
#             "chart_type": "pie",
#             "dataset_source": dataset_source,
#             "created_utc": datetime.utcnow().isoformat() + "Z",
#             "columns_used": {
#                 "date": date_col,
#                 "category": category_col,
#                 "value": value_col,
#             },
#             "data_filters": {
#                 "month": month_str,
#                 "value_positive_only": True,
#                 "top_k_plus_other": True,
#             },
#             "sampling": {
#                 "attempt": attempt,
#                 "n_categories_raw": context["n_categories_raw"],
#                 "top_k": context["top_k"],
#                 "total_value": context["total"],
#             },
#             "outputs": {
#                 "svg": str(svg_path.as_posix()),
#                 "table_csv": str(table_path.as_posix()),
#             },
#             "style_parameters": {
#                 # keep empty for now; later you’ll sample from your priors
#             },
#         }

#         meta_path = out_root / "meta" / f"{chart_id}.json"
#         save_metadata(meta, meta_path)

#         return meta

#     raise RuntimeError(f"Failed to generate a valid pie chart after {max_tries} attempts.")

In [561]:
# Generator
def generate_pie(
    df: pd.DataFrame,
    out_root: Path,
    dataset_source: str,
    library: str,
    rng_seed: int,
    param_stats: dict,
    max_tries: int = 50,
):
    rng = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("pie")

    for _ in range(max_tries):
        style = sample_pie_style(rng, param_stats)

        sampled = sample_pie_data(df=df, rng=rng, style=style)
        if sampled is None:
            continue

        plot_df, context = sampled

        category_col = context["category_col"]
        value_col = context["value_col"]
        title = make_title(context)

        table_path = out_root / "tables" / library / f"{chart_id}.csv"
        meta_path = out_root / "meta" / library / f"{chart_id}.json"

        plot_df.to_csv(table_path, index=False)

        if library == "altair":
            image_path = out_root / "images" / library / f"{chart_id}.svg"
            chart = render_pie_altair(plot_df, category_col, value_col, title, style, rng=rng)
            save_altair_svg(chart, image_path)

        elif library == "matplotlib":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_pie_matplotlib(plot_df, category_col, value_col, title, style, rng=rng)
            save_matplotlib_png(fig, image_path)

        # elif library == "plotly":
        #     image_path = out_root / "images" / library / f"{chart_id}.html"
        #     fig = render_pie_plotly(plot_df, category_col, value_col, title, style)
        #     save_plotly_html(fig, image_path)
        

        elif library == "plotly":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_pie_plotly(plot_df, category_col, value_col, title, style, rng=rng)
            save_plotly_png(fig, image_path)

        else:
            raise ValueError(f"Unsupported library: {library}")

        meta = {
            "chart_id": chart_id,
            "chart_type": "pie",
            "library": library,
            "image_path": str(image_path),
            "table_path": str(table_path),
            "dataset_source": dataset_source,
            "data_context": context,
            "style": style,
            "created_utc": datetime.utcnow().isoformat() + "Z",
        }

        save_metadata(meta, meta_path)
        return meta

    raise RuntimeError(f"Failed to generate chart after {max_tries} attempts.")

In [562]:
# OLD sanity test 

# # sanity test
# OUT_ROOT = PROJECT / "data" / "generated" / "pie"
# dataset_source = "Warehouse and Retail Sales (your saved file / data.gov link)"

# meta = generate_pie(
#     df=df_train,
#     out_root=OUT_ROOT,
#     dataset_source=dataset_source,
#     rng_seed=123
# )

# meta

In [563]:
# # OLD generate pie charts 
# # make 10 pie charts 

# metas = []

# for i in range(10):
#     meta = generate_pie(
#         df=df_train,
#         out_root=OUT_ROOT,
#         dataset_source="Warehouse and Retail Sales",
#         rng_seed=1000 + i   # different seed per chart
#     )
#     metas.append(meta)

# print("Generated:", len(metas), "charts")
# print("First chart ID:", metas[0]["chart_id"])

In [564]:
generation_plan = {
    "altair": 30,
    "matplotlib": 30,
    "plotly": 30
}

dataset_source = "Warehouse and Retail Sales"
metas = []

seed = 1000

for library, n in generation_plan.items():
    for _ in range(n):
        meta = generate_pie(
            df=df_train,
            out_root=OUT_ROOT,
            dataset_source=dataset_source,
            library=library,
            rng_seed=seed,
            param_stats=PARAM_STATS,
        )
        metas.append(meta)
        seed += 1

print(f"Generated {len(metas)} charts total.")
pd.DataFrame(metas)[["chart_id", "library", "image_path"]].head()

Generated 90 charts total.


,chart_id,library,image_path
0,pie_20260326T163620_be40a299,altair,C:\Users\Michelle\I2R\outputs\generated\pie\im...
1,pie_20260326T163621_94a96d30,altair,C:\Users\Michelle\I2R\outputs\generated\pie\im...
2,pie_20260326T163621_9c122978,altair,C:\Users\Michelle\I2R\outputs\generated\pie\im...
3,pie_20260326T163622_08612866,altair,C:\Users\Michelle\I2R\outputs\generated\pie\im...
4,pie_20260326T163622_54e69164,altair,C:\Users\Michelle\I2R\outputs\generated\pie\im...



\current sampling: you always do month → group by ITEM TYPE → sum RETAIL SALES. Since ITEM TYPE has only a few stable categories (BEER/LIQUOR/WINE/…), the shares barely change month to month, so the pies look almost identical.